# 04. R(t) heatmaps under mobility-reduction scenarios

This notebook reads seasonal \(R(t)\) files and generates heatmaps for observed mobility and 10%, 20%, 30%, 40%, and 50% mobility-reduction scenarios. It supports manuscript Figure 7 and Supplementary Figures S6-S9.


In [ ]:
# Repository path setup
# This cell makes the notebook runnable from either the repository root or the notebooks/ directory.
from pathlib import Path
import os


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    if current.name == "notebooks":
        return current.parent
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

for directory in [
    "data/metro",
    "data/NHIS/2016~2017",
    "data/mobility_factor/2016~2017",
    "data/Rt/2016~2017",
    "data/Rt/2017~2018",
    "data/Rt/2018~2019",
    "data/Rt/2022~2023",
    "figures/mobility_factor",
    "figures/2016~2017",
    "figures/HeatMap",
    "figures/validation",
    "results/validation",
    "results/tables",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


HEATMAP_BORDER_WIDTH = 0.5
CBAR_BORDER_WIDTH = 0.5
TICK_WIDTH = 0.8
RT1_LINE_WIDTH = 0.5

# ==============================
# 1. Path setting
# ==============================
DATA_DIR = Path("data/Rt")
SAVE_PATH = Path("figures/HeatMap/Seoul_Rt_heatmaps.eps")

# ==============================
# 2. File and column setting
# ==============================
season_files = {
    "16-17": DATA_DIR / "2016~2017" / "Seoul_Rt(2016~2017).csv",
    "17-18": DATA_DIR / "2017~2018" / "Seoul_Rt(2017~2018).csv",
    "18-19": DATA_DIR / "2018~2019" / "Seoul_Rt(2018~2019).csv",
    "22-23": DATA_DIR / "2022~2023" / "Seoul_Rt(2022~2023).csv",
}

season_start_dates = {
    "16-17": "2016-09-11",
    "17-18": "2017-09-11",
    "18-19": "2018-09-11",
    "22-23": "2022-09-11",
}

rt_cols = [
    "Seoul_Rt",
    "Seoul_Rt_1",
    "Seoul_Rt_2",
    "Seoul_Rt_3",
    "Seoul_Rt_4",
    "Seoul_Rt_5",
]

month_labels = ["Sep", "Nov", "Jan", "Mar", "May", "Jul", "Aug"]

# ==============================
# 3. Style setting
# ==============================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

# Attached-image-like colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_rt",
    [
        (0.00, "#42579A"),   # deep blue
        (0.20, "#7C9AC6"),
        (0.50, "#D8D2CA"),   # gray-beige center
        (0.80, "#D98B72"),
        (1.00, "#A6353C"),   # deep red
    ],
    N=256
)

# Rt scale
vmin, vcenter, vmax = 0.0, 1.0, 2.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ==============================
# 4. Load CSV
# ==============================
season_data = {}
max_len = 0

for season, file_path in season_files.items():

    if not file_path.exists():
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {file_path}")

    print("Loading:", season, file_path)

    df = pd.read_csv(file_path)

    # date column 확인
    if "date" not in df.columns:
        raise ValueError(f"'date' 컬럼이 없습니다: {file_path}")

    # date 형식 변환 및 정렬
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # ==============================
    # 각 시즌별 시작일 설정
    # ==============================
    start_date = pd.to_datetime(season_start_dates[season])

    df = df[df["date"] >= start_date].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{season}에서 {start_date.date()} 이후 데이터가 없습니다.")

    # Rt columns smoothing
    for col in rt_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].rolling(window=7, center=True, min_periods=1).mean()

    season_data[season] = df
    max_len = max(max_len, len(df))

    print(f"{season}: {df['date'].min().date()} ~ {df['date'].max().date()}, n={len(df)}")

# Month tick positions
month_positions = np.linspace(0, max_len - 1, len(month_labels))

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.88, top=0.86, bottom=0.12, wspace=0.15, hspace=0.20)

season_labels = list(season_files.keys())

for i, col in enumerate(rt_cols):
    ax = axes[i]

    # build matrix: rows=season, cols=time
    mat = np.full((len(season_labels), max_len), np.nan)

    for r, season in enumerate(season_labels):
        vals = season_data[season][col].to_numpy()
        mat[r, :len(vals)] = vals

    im = ax.imshow(
        mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="bicubic"
    )

    # heatmap panel border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(HEATMAP_BORDER_WIDTH)
        spine.set_edgecolor("black")
    
    ax.tick_params(
        axis="both",
        width=TICK_WIDTH,
        length=3
    )

    ax.set_title(col, pad=14)

    # y-axis
    if i % 3 == 0:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels(season_labels)
        ax.set_ylabel("Season")
    else:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels([])

    # x-axis
    if i >= 3:
        ax.set_xticks(month_positions)
        ax.set_xticklabels(month_labels)
        ax.set_xlabel("Month")
    else:
        ax.set_xticks(month_positions)
        ax.set_xticklabels([])

    ax.tick_params(length=3)

# ==============================
# 6. Colorbar
# ==============================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.60])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r"${\mathcal{R}}(t)$", rotation=90, labelpad=15, fontsize=18)
cbar.set_ticks([0.0, 0.5, 1.0, 1.2, 1.4, 1.6, 1.8 , 2.0])
cbar.ax.tick_params(labelsize=12)

# colorbar border
cbar.outline.set_linewidth(CBAR_BORDER_WIDTH)
cbar.outline.set_edgecolor("black")

# Rt = 1 reference line
cbar.ax.axhline(
    vcenter,
    color="black",
    linewidth=RT1_LINE_WIDTH
)

# reference line at Rt = 1
cbar.ax.axhline(vcenter, color="black", linewidth=1.6)

# ==============================
# 7. Figure title
# ==============================
fig.suptitle(
    r"Heatmaps of the instantaneous reproduction number (${\mathcal{R}}(t)$)",
    fontsize=24,
    y=0.96
)

# ==============================
# 8. Save and show
# ==============================
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


HEATMAP_BORDER_WIDTH = 0.5
CBAR_BORDER_WIDTH = 0.5
TICK_WIDTH = 0.8
RT1_LINE_WIDTH = 0.5

# ==============================
# 1. Path setting
# ==============================
DATA_DIR = Path("data/Rt")
SAVE_PATH = Path("figures/HeatMap/Busan_Rt_heatmaps.eps")

# ==============================
# 2. File and column setting
# ==============================
season_files = {
    "16-17": DATA_DIR / "2016~2017" / "Busan_Rt(2016~2017).csv",
    "17-18": DATA_DIR / "2017~2018" / "Busan_Rt(2017~2018).csv",
    "18-19": DATA_DIR / "2018~2019" / "Busan_Rt(2018~2019).csv",
    "22-23": DATA_DIR / "2022~2023" / "Busan_Rt(2022~2023).csv",
}

season_start_dates = {
    "16-17": "2016-09-11",
    "17-18": "2017-09-11",
    "18-19": "2018-09-11",
    "22-23": "2022-09-11",
}

rt_cols = [
    "Busan_Rt",
    "Busan_Rt_1",
    "Busan_Rt_2",
    "Busan_Rt_3",
    "Busan_Rt_4",
    "Busan_Rt_5",
]

month_labels = ["Sep", "Nov", "Jan", "Mar", "May", "Jul", "Aug"]

# ==============================
# 3. Style setting
# ==============================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

# Attached-image-like colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_rt",
    [
        (0.00, "#42579A"),   # deep blue
        (0.20, "#7C9AC6"),
        (0.50, "#D8D2CA"),   # gray-beige center
        (0.80, "#D98B72"),
        (1.00, "#A6353C"),   # deep red
    ],
    N=256
)

# Rt scale
vmin, vcenter, vmax = 0.0, 1.0, 2.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ==============================
# 4. Load CSV
# ==============================
season_data = {}
max_len = 0

for season, file_path in season_files.items():

    if not file_path.exists():
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {file_path}")

    print("Loading:", season, file_path)

    df = pd.read_csv(file_path)

    # date column 확인
    if "date" not in df.columns:
        raise ValueError(f"'date' 컬럼이 없습니다: {file_path}")

    # date 형식 변환 및 정렬
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # ==============================
    # 각 시즌별 시작일 설정
    # ==============================
    start_date = pd.to_datetime(season_start_dates[season])

    df = df[df["date"] >= start_date].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{season}에서 {start_date.date()} 이후 데이터가 없습니다.")

    # Rt columns smoothing
    for col in rt_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].rolling(window=7, center=True, min_periods=1).mean()

    season_data[season] = df
    max_len = max(max_len, len(df))

    print(f"{season}: {df['date'].min().date()} ~ {df['date'].max().date()}, n={len(df)}")

# Month tick positions
month_positions = np.linspace(0, max_len - 1, len(month_labels))

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.88, top=0.86, bottom=0.12, wspace=0.15, hspace=0.20)

season_labels = list(season_files.keys())

for i, col in enumerate(rt_cols):
    ax = axes[i]

    # build matrix: rows=season, cols=time
    mat = np.full((len(season_labels), max_len), np.nan)

    for r, season in enumerate(season_labels):
        vals = season_data[season][col].to_numpy()
        mat[r, :len(vals)] = vals

    im = ax.imshow(
        mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="bicubic"
    )

    # heatmap panel border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(HEATMAP_BORDER_WIDTH)
        spine.set_edgecolor("black")
    
    ax.tick_params(
        axis="both",
        width=TICK_WIDTH,
        length=3
    )

    ax.set_title(col, pad=14)

    # y-axis
    if i % 3 == 0:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels(season_labels)
        ax.set_ylabel("Season")
    else:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels([])

    # x-axis
    if i >= 3:
        ax.set_xticks(month_positions)
        ax.set_xticklabels(month_labels)
        ax.set_xlabel("Month")
    else:
        ax.set_xticks(month_positions)
        ax.set_xticklabels([])

    ax.tick_params(length=3)

# ==============================
# 6. Colorbar
# ==============================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.60])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r"${\mathcal{R}}(t)$", rotation=90, labelpad=15, fontsize=18)
cbar.set_ticks([0.0, 0.5, 1.0, 1.2, 1.4, 1.6, 1.8 , 2.0])
cbar.ax.tick_params(labelsize=12)

# colorbar border
cbar.outline.set_linewidth(CBAR_BORDER_WIDTH)
cbar.outline.set_edgecolor("black")

# Rt = 1 reference line
cbar.ax.axhline(
    vcenter,
    color="black",
    linewidth=RT1_LINE_WIDTH
)

# reference line at Rt = 1
cbar.ax.axhline(vcenter, color="black", linewidth=1.6)

# ==============================
# 7. Figure title
# ==============================
fig.suptitle(
    r"Heatmaps of the instantaneous reproduction number (${\mathcal{R}}(t)$)",
    fontsize=24,
    y=0.96
)

# ==============================
# 8. Save and show
# ==============================
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


HEATMAP_BORDER_WIDTH = 0.5
CBAR_BORDER_WIDTH = 0.5
TICK_WIDTH = 0.8
RT1_LINE_WIDTH = 0.5

# ==============================
# 1. Path setting
# ==============================
DATA_DIR = Path("data/Rt")
SAVE_PATH = Path("figures/HeatMap/Daegu_Rt_heatmaps.eps")

# ==============================
# 2. File and column setting
# ==============================
season_files = {
    "16-17": DATA_DIR / "2016~2017" / "Daegu_Rt(2016~2017).csv",
    "17-18": DATA_DIR / "2017~2018" / "Daegu_Rt(2017~2018).csv",
    "18-19": DATA_DIR / "2018~2019" / "Daegu_Rt(2018~2019).csv",
    "22-23": DATA_DIR / "2022~2023" / "Daegu_Rt(2022~2023).csv",
}

season_start_dates = {
    "16-17": "2016-09-11",
    "17-18": "2017-09-11",
    "18-19": "2018-09-11",
    "22-23": "2022-09-11",
}

rt_cols = [
    "Daegu_Rt",
    "Daegu_Rt_1",
    "Daegu_Rt_2",
    "Daegu_Rt_3",
    "Daegu_Rt_4",
    "Daegu_Rt_5",
]

month_labels = ["Sep", "Nov", "Jan", "Mar", "May", "Jul", "Aug"]

# ==============================
# 3. Style setting
# ==============================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

# Attached-image-like colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_rt",
    [
        (0.00, "#42579A"),   # deep blue
        (0.20, "#7C9AC6"),
        (0.50, "#D8D2CA"),   # gray-beige center
        (0.80, "#D98B72"),
        (1.00, "#A6353C"),   # deep red
    ],
    N=256
)

# Rt scale
vmin, vcenter, vmax = 0.0, 1.0, 2.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ==============================
# 4. Load CSV
# ==============================
season_data = {}
max_len = 0

for season, file_path in season_files.items():

    if not file_path.exists():
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {file_path}")

    print("Loading:", season, file_path)

    df = pd.read_csv(file_path)

    # date column 확인
    if "date" not in df.columns:
        raise ValueError(f"'date' 컬럼이 없습니다: {file_path}")

    # date 형식 변환 및 정렬
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # ==============================
    # 각 시즌별 시작일 설정
    # ==============================
    start_date = pd.to_datetime(season_start_dates[season])

    df = df[df["date"] >= start_date].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{season}에서 {start_date.date()} 이후 데이터가 없습니다.")

    # Rt columns smoothing
    for col in rt_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].rolling(window=7, center=True, min_periods=1).mean()

    season_data[season] = df
    max_len = max(max_len, len(df))

    print(f"{season}: {df['date'].min().date()} ~ {df['date'].max().date()}, n={len(df)}")

# Month tick positions
month_positions = np.linspace(0, max_len - 1, len(month_labels))

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.88, top=0.86, bottom=0.12, wspace=0.15, hspace=0.20)

season_labels = list(season_files.keys())

for i, col in enumerate(rt_cols):
    ax = axes[i]

    # build matrix: rows=season, cols=time
    mat = np.full((len(season_labels), max_len), np.nan)

    for r, season in enumerate(season_labels):
        vals = season_data[season][col].to_numpy()
        mat[r, :len(vals)] = vals

    im = ax.imshow(
        mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="bicubic"
    )

    # heatmap panel border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(HEATMAP_BORDER_WIDTH)
        spine.set_edgecolor("black")
    
    ax.tick_params(
        axis="both",
        width=TICK_WIDTH,
        length=3
    )

    ax.set_title(col, pad=14)

    # y-axis
    if i % 3 == 0:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels(season_labels)
        ax.set_ylabel("Season")
    else:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels([])

    # x-axis
    if i >= 3:
        ax.set_xticks(month_positions)
        ax.set_xticklabels(month_labels)
        ax.set_xlabel("Month")
    else:
        ax.set_xticks(month_positions)
        ax.set_xticklabels([])

    ax.tick_params(length=3)

# ==============================
# 6. Colorbar
# ==============================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.60])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r"${\mathcal{R}}(t)$", rotation=90, labelpad=15, fontsize=18)
cbar.set_ticks([0.0, 0.5, 1.0, 1.2, 1.4, 1.6, 1.8 , 2.0])
cbar.ax.tick_params(labelsize=12)

# colorbar border
cbar.outline.set_linewidth(CBAR_BORDER_WIDTH)
cbar.outline.set_edgecolor("black")

# Rt = 1 reference line
cbar.ax.axhline(
    vcenter,
    color="black",
    linewidth=RT1_LINE_WIDTH
)

# reference line at Rt = 1
cbar.ax.axhline(vcenter, color="black", linewidth=1.6)

# ==============================
# 7. Figure title
# ==============================
fig.suptitle(
    r"Heatmaps of the instantaneous reproduction number (${\mathcal{R}}(t)$)",
    fontsize=24,
    y=0.96
)

# ==============================
# 8. Save and show
# ==============================
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


HEATMAP_BORDER_WIDTH = 0.5
CBAR_BORDER_WIDTH = 0.5
TICK_WIDTH = 0.8
RT1_LINE_WIDTH = 0.5

# ==============================
# 1. Path setting
# ==============================
DATA_DIR = Path("data/Rt")
SAVE_PATH = Path("figures/HeatMap/Daejeon_Rt_heatmaps.eps")

# ==============================
# 2. File and column setting
# ==============================
season_files = {
    "16-17": DATA_DIR / "2016~2017" / "Daejeon_Rt(2016~2017).csv",
    "17-18": DATA_DIR / "2017~2018" / "Daejeon_Rt(2017~2018).csv",
    "18-19": DATA_DIR / "2018~2019" / "Daejeon_Rt(2018~2019).csv",
    "22-23": DATA_DIR / "2022~2023" / "Daejeon_Rt(2022~2023).csv",
}

season_start_dates = {
    "16-17": "2016-09-11",
    "17-18": "2017-09-11",
    "18-19": "2018-09-11",
    "22-23": "2022-09-11",
}

rt_cols = [
    "Daejeon_Rt",
    "Daejeon_Rt_1",
    "Daejeon_Rt_2",
    "Daejeon_Rt_3",
    "Daejeon_Rt_4",
    "Daejeon_Rt_5",
]

month_labels = ["Sep", "Nov", "Jan", "Mar", "May", "Jul", "Aug"]

# ==============================
# 3. Style setting
# ==============================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

# Attached-image-like colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_rt",
    [
        (0.00, "#42579A"),   # deep blue
        (0.20, "#7C9AC6"),
        (0.50, "#D8D2CA"),   # gray-beige center
        (0.80, "#D98B72"),
        (1.00, "#A6353C"),   # deep red
    ],
    N=256
)

# Rt scale
vmin, vcenter, vmax = 0.0, 1.0, 2.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ==============================
# 4. Load CSV
# ==============================
season_data = {}
max_len = 0

for season, file_path in season_files.items():

    if not file_path.exists():
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {file_path}")

    print("Loading:", season, file_path)

    df = pd.read_csv(file_path)

    # date column 확인
    if "date" not in df.columns:
        raise ValueError(f"'date' 컬럼이 없습니다: {file_path}")

    # date 형식 변환 및 정렬
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # ==============================
    # 각 시즌별 시작일 설정
    # ==============================
    start_date = pd.to_datetime(season_start_dates[season])

    df = df[df["date"] >= start_date].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{season}에서 {start_date.date()} 이후 데이터가 없습니다.")

    # Rt columns smoothing
    for col in rt_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].rolling(window=7, center=True, min_periods=1).mean()

    season_data[season] = df
    max_len = max(max_len, len(df))

    print(f"{season}: {df['date'].min().date()} ~ {df['date'].max().date()}, n={len(df)}")

# Month tick positions
month_positions = np.linspace(0, max_len - 1, len(month_labels))

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.88, top=0.86, bottom=0.12, wspace=0.15, hspace=0.20)

season_labels = list(season_files.keys())

for i, col in enumerate(rt_cols):
    ax = axes[i]

    # build matrix: rows=season, cols=time
    mat = np.full((len(season_labels), max_len), np.nan)

    for r, season in enumerate(season_labels):
        vals = season_data[season][col].to_numpy()
        mat[r, :len(vals)] = vals

    im = ax.imshow(
        mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="bicubic"
    )

    # heatmap panel border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(HEATMAP_BORDER_WIDTH)
        spine.set_edgecolor("black")
    
    ax.tick_params(
        axis="both",
        width=TICK_WIDTH,
        length=3
    )

    ax.set_title(col, pad=14)

    # y-axis
    if i % 3 == 0:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels(season_labels)
        ax.set_ylabel("Season")
    else:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels([])

    # x-axis
    if i >= 3:
        ax.set_xticks(month_positions)
        ax.set_xticklabels(month_labels)
        ax.set_xlabel("Month")
    else:
        ax.set_xticks(month_positions)
        ax.set_xticklabels([])

    ax.tick_params(length=3)

# ==============================
# 6. Colorbar
# ==============================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.60])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r"${\mathcal{R}}(t)$", rotation=90, labelpad=15, fontsize=18)
cbar.set_ticks([0.0, 0.5, 1.0, 1.2, 1.4, 1.6, 1.8 , 2.0])
cbar.ax.tick_params(labelsize=12)

# colorbar border
cbar.outline.set_linewidth(CBAR_BORDER_WIDTH)
cbar.outline.set_edgecolor("black")

# Rt = 1 reference line
cbar.ax.axhline(
    vcenter,
    color="black",
    linewidth=RT1_LINE_WIDTH
)

# reference line at Rt = 1
cbar.ax.axhline(vcenter, color="black", linewidth=1.6)

# ==============================
# 7. Figure title
# ==============================
fig.suptitle(
    r"Heatmaps of the instantaneous reproduction number (${\mathcal{R}}(t)$)",
    fontsize=24,
    y=0.96
)

# ==============================
# 8. Save and show
# ==============================
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


HEATMAP_BORDER_WIDTH = 0.5
CBAR_BORDER_WIDTH = 0.5
TICK_WIDTH = 0.8
RT1_LINE_WIDTH = 0.5

# ==============================
# 1. Path setting
# ==============================
DATA_DIR = Path("data/Rt")
SAVE_PATH = Path("figures/HeatMap/Gwangju_Rt_heatmaps.eps")

# ==============================
# 2. File and column setting
# ==============================
season_files = {
    "16-17": DATA_DIR / "2016~2017" / "Gwangju_Rt(2016~2017).csv",
    "17-18": DATA_DIR / "2017~2018" / "Gwangju_Rt(2017~2018).csv",
    "18-19": DATA_DIR / "2018~2019" / "Gwangju_Rt(2018~2019).csv",
    "22-23": DATA_DIR / "2022~2023" / "Gwangju_Rt(2022~2023).csv",
}

season_start_dates = {
    "16-17": "2016-09-11",
    "17-18": "2017-09-11",
    "18-19": "2018-09-11",
    "22-23": "2022-09-11",
}

rt_cols = [
    "Gwangju_Rt",
    "Gwangju_Rt_1",
    "Gwangju_Rt_2",
    "Gwangju_Rt_3",
    "Gwangju_Rt_4",
    "Gwangju_Rt_5",
]

month_labels = ["Sep", "Nov", "Jan", "Mar", "May", "Jul", "Aug"]

# ==============================
# 3. Style setting
# ==============================
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

# Attached-image-like colormap
cmap = LinearSegmentedColormap.from_list(
    "custom_rt",
    [
        (0.00, "#42579A"),   # deep blue
        (0.20, "#7C9AC6"),
        (0.50, "#D8D2CA"),   # gray-beige center
        (0.80, "#D98B72"),
        (1.00, "#A6353C"),   # deep red
    ],
    N=256
)

# Rt scale
vmin, vcenter, vmax = 0.0, 1.0, 2.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

# ==============================
# 4. Load CSV
# ==============================
season_data = {}
max_len = 0

for season, file_path in season_files.items():

    if not file_path.exists():
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {file_path}")

    print("Loading:", season, file_path)

    df = pd.read_csv(file_path)

    # date column 확인
    if "date" not in df.columns:
        raise ValueError(f"'date' 컬럼이 없습니다: {file_path}")

    # date 형식 변환 및 정렬
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # ==============================
    # 각 시즌별 시작일 설정
    # ==============================
    start_date = pd.to_datetime(season_start_dates[season])

    df = df[df["date"] >= start_date].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{season}에서 {start_date.date()} 이후 데이터가 없습니다.")

    # Rt columns smoothing
    for col in rt_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].rolling(window=7, center=True, min_periods=1).mean()

    season_data[season] = df
    max_len = max(max_len, len(df))

    print(f"{season}: {df['date'].min().date()} ~ {df['date'].max().date()}, n={len(df)}")

# Month tick positions
month_positions = np.linspace(0, max_len - 1, len(month_labels))

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

fig.subplots_adjust(left=0.08, right=0.88, top=0.86, bottom=0.12, wspace=0.15, hspace=0.20)

season_labels = list(season_files.keys())

for i, col in enumerate(rt_cols):
    ax = axes[i]

    # build matrix: rows=season, cols=time
    mat = np.full((len(season_labels), max_len), np.nan)

    for r, season in enumerate(season_labels):
        vals = season_data[season][col].to_numpy()
        mat[r, :len(vals)] = vals

    im = ax.imshow(
        mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="bicubic"
    )

    # heatmap panel border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(HEATMAP_BORDER_WIDTH)
        spine.set_edgecolor("black")
    
    ax.tick_params(
        axis="both",
        width=TICK_WIDTH,
        length=3
    )

    ax.set_title(col, pad=14)

    # y-axis
    if i % 3 == 0:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels(season_labels)
        ax.set_ylabel("Season")
    else:
        ax.set_yticks(range(len(season_labels)))
        ax.set_yticklabels([])

    # x-axis
    if i >= 3:
        ax.set_xticks(month_positions)
        ax.set_xticklabels(month_labels)
        ax.set_xlabel("Month")
    else:
        ax.set_xticks(month_positions)
        ax.set_xticklabels([])

    ax.tick_params(length=3)

# ==============================
# 6. Colorbar
# ==============================
cbar_ax = fig.add_axes([0.90, 0.18, 0.02, 0.60])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label(r"${\mathcal{R}}(t)$", rotation=90, labelpad=15, fontsize=18)
cbar.set_ticks([0.0, 0.5, 1.0, 1.2, 1.4, 1.6, 1.8 , 2.0])
cbar.ax.tick_params(labelsize=12)

# colorbar border
cbar.outline.set_linewidth(CBAR_BORDER_WIDTH)
cbar.outline.set_edgecolor("black")

# Rt = 1 reference line
cbar.ax.axhline(
    vcenter,
    color="black",
    linewidth=RT1_LINE_WIDTH
)

# reference line at Rt = 1
cbar.ax.axhline(vcenter, color="black", linewidth=1.6)

# ==============================
# 7. Figure title
# ==============================
fig.suptitle(
    r"Heatmaps of the instantaneous reproduction number (${\mathcal{R}}(t)$)",
    fontsize=24,
    y=0.96
)

# ==============================
# 8. Save and show
# ==============================
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SAVE_PATH, dpi=600, bbox_inches="tight")
plt.show()